In [ ]:
# Competition-Solution/notebooks/train/02_model_training/03_train_model_training_multi_task_learning.ipynb

##### Note

This notebook is exploratory and was not used in the final competition submission. The multi-task learning approach was not fully implemented due to computational and time constraints.


### Codabench Baseline Metrics

The Codabench contest provided baseline results to offer a starting benchmark. Their baselines were formed based on the full training dataset. The approaches and their reported Macro-F1 scores for each subtask are as follows:

*   **Subtask 1: Call2Action (Binary Classification)**
    *   **Baseline Approach:** Gradient-boosting classifier with SentenceBert embeddings and tweet polarity, using undersampling.
    *   **Reported Macro-F1 Score:** 0.59

*   **Subtask 2: Attacks on the Democratic Basic Order (FDGO) (Multi-class Classification)**
    *   **Baseline Approach:** Linear Support Vector Machine (SVM) with TF-IDF weighted bag-of-phrases (unigrams and bigrams), using a cost-sensitive SVM.
    *   **Reported Macro-F1 Score:** 0.47

*   **Subtask 3: Violence Detection (Binary Classification)**
    *   **Baseline Approach:** Large Language Model Qwen2.5 (32 billion parameters) in a few-shot scenario.
    *   **Reported Macro-F1 Score:** 0.69

These scores serve as an initial comparison point for the subsequent approaches and training runs.
These allow us to ballpark the performance of different approaches and motivate why we use an encoder-based transformer model for our main attempts.

---

##### <b>Imports</b>

In [ ]:
import sys
from collections import Counter
from pathlib import Path
from rich.console import Console
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

import wandb
import torch
import torch.nn as nn

# Hugging Face Imports
from transformers import (
  AutoProcessor,
  Trainer,
  TrainingArguments,
  EarlyStoppingCallback,
  AutoModel,
  PreTrainedModel,
  AutoConfig,
  DataCollatorWithPadding
)
from datasets import load_from_disk, Dataset, DatasetDict
import evaluate

console = Console()

# Path to this notebook
notebook_dir = Path.cwd()

# Project root directory
project_root_dir = notebook_dir.parent.parent.parent

# Source path
src_path = project_root_dir / "src"
sys.path.append(str(src_path))

console.print(f"Project root: {project_root_dir}", style="cyan")
console.print(f"Source path: {src_path}", style="cyan")
console.print(f"Source path exists: {src_path.exists()}", style="cyan")

# Local imports
import wandb_utils
import config_utils

### **Experiment Configuration**

In [ ]:
# MTL Configuration - No subtask selection needed since we train all tasks
default_config_choices = {
    "CHOSEN_EXPERIMENT_FILE_NAME": "ModernGBERT_MTL_train.yaml",
    "CHOSEN_DATASET_MODE": "train",
    "RUN_VERSION_TAG": "v1",
    "LOG_TO_WANDB": True
}

# Initializes default config choices
global_config_choices = default_config_choices.copy()

console.print("Initial/Default Configuration Choices:", style="bold yellow")
for config_option, config_choice in global_config_choices.items():
    console.print(f"  {config_option}: {config_choice}")

In [ ]:

# Simplified options for MTL
experiment_options = [
    ("Modern-GBERT · MTL train", "ModernGBERT_MTL_train.yaml"),
]

# Simplified options for dataset mode
dataset_mode_options = [
    ("Train", "train"),
]

# Form Widgets

# Experiment Dropdown
dd_experiment = widgets.Dropdown(
    options=experiment_options,
    value=global_config_choices['CHOSEN_EXPERIMENT_FILE_NAME'],
    description="MTL Experiment:",
    layout=widgets.Layout(width="320px"),
    style={"description_width": "90px"}
)

# Dataset Mode Dropdown
dd_mode = widgets.Dropdown(
    options=dataset_mode_options,
    value=global_config_choices['CHOSEN_DATASET_MODE'],
    description="Dataset mode:",
    layout=widgets.Layout(width="320px"),
    style={"description_width": "90px"}
)

# Run Version Text Input
txt_version = widgets.Text(
    value=global_config_choices['RUN_VERSION_TAG'],
    description="Run tag:",
    placeholder="e.g. v1",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="200px")
)

# Log to W&B Checkbox
log_to_wandb = widgets.Checkbox(
    description="Log to W&B",
    value=global_config_choices['LOG_TO_WANDB'],
    layout=widgets.Layout(width="200px"),
    style={"description_width": "90px"}
)

# Save Button
save_btn = widgets.Button(
    description="Save choices",
    button_style="success",
    icon="check"
)

status_out = widgets.Output()

def _on_run_clicked(b):
    global global_config_choices
    global_config_choices = {
        "CHOSEN_EXPERIMENT_FILE_NAME": dd_experiment.value,
        "CHOSEN_DATASET_MODE": dd_mode.value,
        "RUN_VERSION_TAG": txt_version.value,
        "LOG_TO_WANDB": log_to_wandb.value
    }
    with status_out:
        clear_output(wait=True)
        console.print("Config saved:", global_config_choices, style="green")

save_btn.on_click(_on_run_clicked)

# Builds the form
form = widgets.VBox([
    widgets.HTML("<h4 style='margin:0 0 8px 0'>Configure experiment</h4>"),
    dd_experiment,
    dd_mode,
    txt_version,
    log_to_wandb,
    save_btn,
    status_out
])

display(form)

In [ ]:
# Path to the configs directory
config_base_dir_nb = project_root_dir / "configs"

# Defines the paths to the global and MTL experiment configs
base_cfg_path = config_base_dir_nb / "base.yaml" # Global base config
experiment_cfg_path = config_base_dir_nb / "mtl" / global_config_choices['CHOSEN_EXPERIMENT_FILE_NAME'] # MTL experiment config

# Loads the global and experiment configs (no subtask config for MTL)
cfg = config_utils.load_config(
  base_config_path=base_cfg_path,
  subtask_config_path=None,  # No subtask config for MTL
  experiment_config_path=experiment_cfg_path
)

# Raises an error if one of the config files is not found
if not cfg:
  raise ValueError(f'Configuration files could not be loaded. Please check the config YAML files in the \"configs\" directory and widget selections:\n'
                   f"    Global base config: {base_cfg_path}\n"
                   f"    MTL experiment config: {experiment_cfg_path}")

console.print(f"Successfully loaded and merged configurations for: \n"
              f"mtl / {global_config_choices['CHOSEN_EXPERIMENT_FILE_NAME']} "
              f"(Mode: {global_config_choices['CHOSEN_DATASET_MODE']})", style="green")

##### <b>Loading the HF datasets</b>

In [ ]:
### Creating the Combined MTL Dataset
console.print("\n[bold yellow]Creating Combined MTL Dataset...[/bold yellow]")

# Loads the individual train datasets
c2a_train_dataset = load_from_disk("../../../data/processed/c2a/c2a_hf_dataset_train")
dbo_train_dataset = load_from_disk("../../../data/processed/dbo/dbo_hf_dataset_train") 
vio_train_dataset = load_from_disk("../../../data/processed/vio/vio_hf_dataset_train")

# Loads the test datasets
c2a_test_dataset = load_from_disk("../../../data/processed/c2a/c2a_hf_dataset_test")
dbo_test_dataset = load_from_disk("../../../data/processed/dbo/dbo_hf_dataset_test")
vio_test_dataset = load_from_disk("../../../data/processed/vio/vio_hf_dataset_test")

# Creates the MTL split
def create_mtl_split(datasets_dict):
    combined = []
    for task_id, dataset in datasets_dict.items():
        for sample in dataset:
            new_sample = {
                'description': sample['description'],
                'id': sample['id'],
                'c2a_labels': sample['C2A'] if task_id == 'c2a' else -100,
                'dbo_labels': sample['DBO'] if task_id == 'dbo' else -100,
                'vio_labels': sample['VIO'] if task_id == 'vio' else -100,
            }
            combined.append(new_sample)
    return Dataset.from_list(combined).shuffle(seed=42)

# Creates the MTL HF dataset dict
mtl_dataset = DatasetDict({
    'train': create_mtl_split({
        'c2a': c2a_train_dataset['train'],
        'dbo': dbo_train_dataset['train'], 
        'vio': vio_train_dataset['train']
    }),
    'validation': create_mtl_split({
        'c2a': c2a_train_dataset['validation'],
        'dbo': dbo_train_dataset['validation'],
        'vio': vio_train_dataset['validation']
    }),
    'test': create_mtl_split({
        'c2a': c2a_test_dataset,
        'dbo': dbo_test_dataset,
        'vio': vio_test_dataset
    })
})

# Saves the combined MTL dataset
mtl_dataset.save_to_disk("../../../data/processed/mtl/mtl_hf_dataset_train")
console.print("MTL dataset saved to: data/processed/mtl/mtl_hf_dataset_train", style="green")

In [ ]:
# Defines the path to the processed data directory
processed_data_root_dir = project_root_dir / cfg['paths']['processed_data_dir_name']
console.print(f"Processed Data Root Directory: '{processed_data_root_dir}'", style="bold")

# Defines the suffix of the dataset (e.g. dbo/dbo_hf_dataset)
dataset_load_path_suffix = cfg['dataset_details']['hf_dataset_path_suffix']
console.print(f"Dataset Load Path Suffix: '{dataset_load_path_suffix}'", style="bold")

# Constructs the full path to the dataset by joining the processed data root directory and the dataset load path suffix
full_dataset_load_path = processed_data_root_dir / dataset_load_path_suffix

# Prints the full path to the dataset
console.print(f"Loading MTL dataset from: '{full_dataset_load_path}'", style="green")

# Tries to load the dataset from the full path
try:
    raw_dataset = load_from_disk(str(full_dataset_load_path))
    console.print(f"Successfully loaded MTL dataset: \n{raw_dataset}", style="green")
except FileNotFoundError:
    console.print(f"ERROR: MTL dataset not found at {full_dataset_load_path}. Please run preprocessing first.", style="bold red")
    raise
except Exception as e:
    console.print(f"ERROR: Could not load MTL dataset from {full_dataset_load_path}: {e}", style="bold red")
    raise

In [ ]:
# Multi-Task Focal Loss implementation
# Extends the original class weights + focal loss to handle multiple tasks simultaneously
# Each task gets its own focal loss calculation with task-specific class weights
# Kendall uncertainty weighting automatically balances the tasks during training
# https://link.springer.com/article/10.1023/A:1007379606734

# Original Focal Loss Formula: FL(p_t) = -α_t * (1 - p_t)^γ * log(p_t)
# MTL Extension: Total_Loss = Σ(1/(2σ²_i) * FL_i + log(σ_i))
# Where σ_i is the learned uncertainty for task i

# Inherits from the PreTrainedModel class to implement multi-task learning
# HF PreTrainedModel Object Docs: https://huggingface.co/docs/transformers/main/en/main_classes/model#transformers.PreTrainedModel
class MultiTaskModel(PreTrainedModel):
    
    # Initializes the MTL model
    def __init__(self, config, mtl_config):
        super().__init__(config)
        self.mtl_config = mtl_config
        
        # Loads the shared encoder
        self.encoder = AutoModel.from_pretrained(config._name_or_path, config=config)
        
        # Gets the hidden size of the encoder
        hidden_size = self.encoder.config.hidden_size
        
        # Creates task-specific classification heads
        self.task_heads = nn.ModuleDict()

        # Loops through the tasks and creates a task-specific classification head for each task
        for task in mtl_config['mtl']['tasks']:
            task_id = task['task_id']
            num_labels = task['num_labels']
            
            # Simple dropout and linear layer for each head
            head = nn.Sequential(
                nn.Dropout(0.1),
                nn.Linear(hidden_size, num_labels)
            )
            self.task_heads[task_id] = head
        
        # Kendall uncertainty parameters (learnable task weights)
        # log_vars represent log(σ²) for each task
        self.log_vars = nn.Parameter(torch.zeros(len(mtl_config['mtl']['tasks'])))
        
    # Forward pass through the model
    def forward(self, input_ids, attention_mask=None, **kwargs):
        # Passes through the shared encoder
        outputs = self.encoder(input_ids, attention_mask=attention_mask)
        
        # Uses the [CLS] token's representation for classification
        # ModernGBERT doesn't have pooler_output, so we use the first token from last_hidden_state
        # (https://huggingface.co/docs/transformers/main/en/model_doc/auto#transformers.AutoModel.last_hidden_state)
        pooler_output = outputs.last_hidden_state[:, 0, :]
        
        # Computes the logits for each task by passing the pooled output through each head
        logits = {}
        for task_id, head in self.task_heads.items():
            logits[task_id] = head(pooler_output)
            
        return {"logits": logits}
    
    # Resizes the token embeddings to match the new number of tokens
    def resize_token_embeddings(self, new_num_tokens, mean_resizing=True):
        """Delegate to the encoder"""
        return self.encoder.resize_token_embeddings(new_num_tokens, mean_resizing=mean_resizing)
    
    # Adds gradient checkpointing support (We had a bug here, which is why I added it manually)
    # Sort of a hotfix
    @property
    def supports_gradient_checkpointing(self):
        """Enable gradient checkpointing support"""
        return True
    
    # Enables gradient checkpointing on the encoder
    def gradient_checkpointing_enable(self, gradient_checkpointing_kwargs=None):
        """Enable gradient checkpointing on the encoder"""
        if hasattr(self.encoder, 'gradient_checkpointing_enable'):
            self.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs)
    
    # Disables gradient checkpointing on the encoder
    def gradient_checkpointing_disable(self):
        """Disable gradient checkpointing on the encoder"""
        if hasattr(self.encoder, 'gradient_checkpointing_disable'):
            self.encoder.gradient_checkpointing_disable()

In [ ]:
# Custom Multi-Task Trainer that implements class weights + focal loss for class imbalance mitigation across multiple tasks

# Multi-task learning allows us to share representations across related tasks since all three subtasks deal with harmful content detection
# The shared encoder learns general patterns of harmful language while task-specific heads specialize on each subtask
# This approach improves generalization by learning from multiple related objectives simultaneously
# We handle class imbalance across all tasks using task-specific class weights and apply focal loss to focus on hard examples

# Focal loss will guide the model to focus on the hard examples across all tasks

# Regular cross entropy loss formula: -log(probability_of_correct_class)
# Focal loss formula: class_weight * difficulty_multiplier * cross_entropy_loss
# difficulty_multiplier = (1 - confidence)^gamma
# Gamma is usually just set to 2.0 as this is what is suggested in the original paper (https://arxiv.org/pdf/1708.02002.pdf)
# Higher gamma values focus more on the hard examples

# Original Formula from paper: FL(p_t) = -α_t * (1 - p_t)^γ * log(p_t)
# Where: 
# p_t = confidence_for_correct_class
# α_t = (alpha) class_weight
# γ = gamma

# Example: Model predicts 0.9 for the correct class, 0.1 for the incorrect class
# confidence_for_correct_class = 0.9
# difficulty_multiplier = (1 - 0.9)^2 = 0.1^2 = 0.01
# cross_entropy_loss = -log(0.9) = 0.105
# focal_loss = class_weight * 0.01 * 0.105 = very small loss
# -> This is an EASY example, so focal loss almost ignores it

# Counter-example: Model predicts 0.3 for the correct class, 0.7 for the incorrect class  
# confidence_for_correct_class = 0.3
# difficulty_multiplier = (1 - 0.3)^2 = 0.7^2 = 0.49
# cross_entropy_loss = -log(0.3) = 1.204
# focal_loss = class_weight * 0.49 * 1.204 = much larger loss
# -> This is a HARD example, so focal loss heavily focuses on it

# As a result, the model spends more time learning from mistakes on hard examples
# rather than wasting time on examples it already gets right

# This is in addition to the class weights, which are used to balance the classes within each task
# The uncertainty weighting then balances the importance of each task automatically

# Inherits from the Trainer class to implement multi-task focal loss
# This will ensure that the model focuses more on the hard examples across all tasks
# and automatically learns how much attention to pay to each task

# HF Trainer Object Docs: https://huggingface.co/docs/transformers/main/en/main_classes/trainer
class MultiTaskFocalLossTrainer(Trainer):
    
    # We pass the task-specific class weights and gamma as instance variables
    def __init__(self, task_class_weights=None, gamma=2.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.task_class_weights = task_class_weights or {}
        self.gamma = gamma

        # Extracts the task IDs from the model config for processing multiple tasks
        self.task_ids = [task['task_id'] for task in self.model.mtl_config['mtl']['tasks']]

    # Overrides the _remove_unused_columns method to keep the MTL label columns
    # The standard Trainer would remove our custom label columns, but we need them for multi-task training
    def _remove_unused_columns(self, dataset, description=None):
        # Standard columns
        signature_columns = ["input_ids", "attention_mask"]
        
        # Adds our MTL label columns to the keep list so they don't get removed
        mtl_label_columns = ["c2a_labels", "dbo_labels", "vio_labels"]
        columns_to_keep = signature_columns + mtl_label_columns
        
        # Only removes columns that are NOT in our keep list
        columns_to_remove = []
        for col in dataset.column_names:
            if col not in columns_to_keep:
                columns_to_remove.append(col)
        
        # Removes the columns that are not in our keep list
        if columns_to_remove:
            dataset = dataset.remove_columns(columns_to_remove)
        
        return dataset

    # Overrides the compute_loss method to implement multi-task focal loss
    # This is called once per batch during training and handles all three tasks simultaneously
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs) # Forward pass through the MTL model
        logits = outputs.get("logits") # Gets the task-specific logits dictionary
        
        total_loss = 0
        
        # Loops through each task and calculates the focal loss separately
        # Each task gets its own focal loss calculation with task-specific class weights
        for task_idx, task_id in enumerate(self.task_ids):
            task_labels = inputs.get(f"{task_id}_labels") # Gets labels for this specific task
            task_logits = logits[task_id] # Gets logits for this specific task
            
            # Ensures task_labels is a tensor and handles single values
            if not isinstance(task_labels, torch.Tensor):
                task_labels = torch.tensor(task_labels, device=task_logits.device)
            
            # Ensures task_labels has the right shape for processing
            if task_labels.dim() == 0:  # Single scalar value
                task_labels = task_labels.unsqueeze(0)  # Make it [1]
            
            # If all labels are -100, these are ignore labels for this task, so we skip them
            valid_mask = task_labels != -100
            if not valid_mask.any():
                continue
                
            # Filters to only valid samples for this task
            valid_labels = task_labels[valid_mask]
            valid_logits = task_logits[valid_mask]
            
            # Converts the logits to probabilities using softmax, essentially normalizing them to sum up to 1
            probabilities = torch.softmax(valid_logits, dim=-1)
            
            # Gets the probability for the correct class for each sample
            # This tells us how confident the model is about the correct answer for this task
            confidence_for_correct_class = probabilities.gather(dim=-1, index=valid_labels.unsqueeze(-1)).squeeze(-1)
            
            # Calculates the difficulty multiplier: (1 - confidence)^gamma
            # High confidence (0.9) -> (1-0.9)^2 = 0.01 (easy, ignore)
            # Low confidence (0.3) -> (1-0.3)^2 = 0.49 (hard, focus)
            difficulty_multiplier = (1 - confidence_for_correct_class) ** self.gamma

            # If class weights are provided for this task, we use them to calculate the focal loss
            if task_id in self.task_class_weights and self.task_class_weights[task_id] is not None:
                # Converts the class weights from Python tensor to PyTorch tensor
                # A tensor is essentially a multi-dimensional array
                # Ensures tensor is on same device (CPU/GPU) as model and uses the appropriate data type 
                # (float16 as we are using half precision for training, so the class weights need to be in half precision too, but logits.dtype infers it from the models logits anyway)
                weight_tensor = self.task_class_weights[task_id].to(
                    dtype=valid_logits.dtype,
                    device=valid_logits.device
                )

                # Creates the loss function with our class weights for this specific task
                # We use reduction="none" instead of the default "mean" to get the loss for each sample in the batch individually
                # We want to discern individual hard tweets from easy tweets and treat them differently
                # Compared to class weights, where each tweet of the same class is treated equally
                loss_fct = nn.CrossEntropyLoss(weight=weight_tensor, reduction='none')
            else:
                # If no class weights are provided for this task, we use the default loss function
                loss_fct = nn.CrossEntropyLoss(reduction='none')

            # Calculates the cross entropy loss for each sample in the batch using the loss function defined above
            # Reshapes tensors to the format CrossEntropyLoss expects:
            # valid_logits -> 2D: [num_valid_samples, num_classes_for_task]  
            # valid_labels -> 1D: [num_valid_samples]
            # The -1 tells PyTorch to infer that dimension automatically
            # (https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
            num_labels_for_task = valid_logits.size(-1)
            cross_entropy_loss = loss_fct(valid_logits.view(-1, num_labels_for_task), valid_labels.view(-1))

            # Applies the focal loss: multiplies cross entropy by difficulty multiplier
            # Again: difficulty_multiplier is (1 - confidence)^gamma
            # cross_entropy_loss is -log(probability_of_correct_class)
            # So, focal_loss = (1 - confidence)^gamma * -log(probability_of_correct_class)
            focal_loss = difficulty_multiplier * cross_entropy_loss
            
            # Takes the mean over the valid samples to get the task loss value
            task_focal_loss = focal_loss.mean()
            
            # Applies Kendall uncertainty weighting for this task
            # This automatically learns to balance tasks based on their uncertainty
            # (https://arxiv.org/pdf/1705.07115)
            # Formula: L_task_weighted = 1/(2σ²) * L_task + log(σ)
            # Where σ is the learned uncertainty parameter for this task
            # Higher uncertainty means the task gets less weight, lower uncertainty means more weight
            precision = torch.exp(-model.log_vars[task_idx])  # 1/σ²
            weighted_task_loss = precision * task_focal_loss + model.log_vars[task_idx]
            
            total_loss += weighted_task_loss
        
        # The final loss is the sum of all weighted task losses
        # Gradient descent expects a single scalar loss value per batch to calculate gradients
        # The total loss represents the combined "wrongness" across all tasks for this batch
        # which is then used to update all model parameters to improve performance on all tasks
        return (total_loss, outputs) if return_outputs else total_loss

In [ ]:
# Calculates the class weights for each task
def calculate_mtl_class_weights(dataset, task_ids):
    task_class_weights = {}
    
    for task_id in task_ids:
        label_column = f"{task_id}_labels"
        
        # Extracts valid labels (not -100) for this task
        valid_labels = [label for label in dataset[label_column] if label != -100]
        
        if not valid_labels:
            console.print(f"WARNING: No valid labels found for task {task_id}", style="yellow")
            continue
            
        # Counter comes from the python standard collections library and counts the occurences of each element in the list
        # e.g. if labels is [0, 0, 1, 1, 2, 2, 2], class_counts will be {0: 2, 1: 2, 2: 3}
        class_counts = Counter(valid_labels)
        num_classes = len(class_counts)
        total_samples = len(valid_labels)

        # Calculates the weights: total_samples / (num_classes * samples_in_class_i)
        class_weights = []

        # Loops through the classes and calculates the weights using the formula above
        for class_id in sorted(class_counts.keys()):
            weight = total_samples / (num_classes * class_counts[class_id])
            class_weights.append(weight)

        task_class_weights[task_id] = class_weights
    
    return task_class_weights

In [ ]:
# Prints the class weights and their distribution for all tasks
def print_mtl_class_weights_info(task_class_weights, dataset, task_ids):
    # Loops through the tasks and prints the class weights and their distribution
    for task_id in task_ids:
        if task_id not in task_class_weights:
            continue
            
        label_column = f"{task_id}_labels"

        # Extracts valid labels (not -100) for this task
        valid_labels = [label for label in dataset[label_column] if label != -100]

        # Counter comes from the python standard collections library and counts the occurences of each element in the list
        # e.g. if labels is [0, 0, 1, 1, 2, 2, 2], class_counts will be {0: 2, 1: 2, 2: 3}
        class_counts = Counter(valid_labels)

        # Gets the total number of samples for this task
        total_samples = len(valid_labels)

        # Gets the class weights for this task
        class_weights = task_class_weights[task_id]

        console.print(f"\n[bold cyan]{task_id.upper()} Task[/bold cyan] Class Distribution and Weights:")
        
        for class_id in sorted(class_counts.keys()):
            weight = class_weights[class_id]
            console.print(f"  Class {class_id}: {class_counts[class_id]} samples, Weight: {weight:.4f}")

        console.print(f"  Total Samples: {total_samples}")
        console.print(f"  Number of Classes: {len(class_counts)}")


In [ ]:
# Model configuration and paths
console.print(f"Model: {cfg['model_checkpoint']}", style="bold")
console.print(f"Model config: {cfg['model_config']}", style="bold")

# Checks for local model first, fallback to HF Hub
local_model_path = project_root_dir / cfg['paths']['base_models_dir_name'] / cfg['model_checkpoint']
if local_model_path.exists():
    model_path = str(local_model_path)
    console.print(f"Found local model at: {model_path}", style="cyan")
else:
    model_path = cfg['model_checkpoint']
    console.print(f"Local model not found, using Hugging Face Hub: {model_path}", style="yellow")

# Loads the model config
try:
    model_config = AutoConfig.from_pretrained(model_path, **cfg['model_config'])
    model_config._name_or_path = model_path
    console.print(f"Successfully loaded model config", style="green")
except Exception as e:
    console.print(f"ERROR: Could not load model config from {model_path}: {e}", style="bold red")
    raise

# Creates the MTL model
try:
    model = MultiTaskModel(model_config, cfg)
    console.print(f"Successfully created MTL model with {len(cfg['mtl']['tasks'])} task heads", style="green")
    
    for task in cfg['mtl']['tasks']:
        console.print(f"  - {task['task_id']}: {task['num_labels']} labels", style="cyan")
        
except Exception as e:
    console.print(f"ERROR: Could not create MTL model: {e}", style="bold red")
    raise

# Moves the model to the GPU if available
if torch.cuda.is_available():
    model.to("cuda")
    console.print("MTL model moved to GPU", style="green")

In [ ]:
console.print(f"Loading AutoProcessor for '{model_path}'", style="green")

processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
processor.add_special_tokens({"additional_special_tokens": cfg['tokenization']['special_tokens']})

# Now this will work because we added the method to our MTL model
model.resize_token_embeddings(len(processor), mean_resizing=True)

console.print(f"Processor loaded with {len(cfg['tokenization']['special_tokens'])} special tokens, total: {len(processor)}", style="cyan")

In [ ]:
# Tokenization function that preserves MTL labels
def tokenize_function(examples):
    tokenized = processor(
        examples[cfg['tokenization']['text_column_name']],
        padding=cfg['tokenization']['padding'],
        truncation=cfg['tokenization']['truncation'],
    )
    
    # Preserves the MTL label columns
    tokenized['c2a_labels'] = examples['c2a_labels']
    tokenized['dbo_labels'] = examples['dbo_labels'] 
    tokenized['vio_labels'] = examples['vio_labels']
    
    # Preserves the id column
    tokenized['id'] = examples['id']
    
    return tokenized

# Applies the tokenization mapping function to the raw dataset in batches
try:
    tokenized_dataset = raw_dataset.map(tokenize_function, batched=True)
    console.print(f"Dataset tokenized successfully: {tokenized_dataset}", style="bold green")
except Exception as e:
    console.print(f"ERROR during dataset tokenization: {e}", style="bold red")
    raise

# Saves the tokenized dataset to disk
save_path = processed_data_root_dir / cfg['dataset_details']['hf_dataset_tokenized_path_suffix']
tokenized_dataset.save_to_disk(str(save_path))
console.print(f"Tokenized MTL dataset saved to: {save_path}", style="green")

In [ ]:
# Calculates the class weights for all tasks
console.print("Calculating class weights for each task...", style="yellow")


task_ids = []
for task in cfg['mtl']['tasks']:
    task_ids.append(task['task_id'])
task_class_weights_raw = calculate_mtl_class_weights(
    tokenized_dataset[cfg['dataset_splits']['train']], 
    task_ids
)

# Converts the class weights to tensors
task_class_weights = {}
for task_id, weights in task_class_weights_raw.items():
    task_class_weights[task_id] = torch.tensor(weights, dtype=torch.float)

# Prints the class weights info
print_mtl_class_weights_info(task_class_weights_raw, tokenized_dataset[cfg['dataset_splits']['train']], task_ids)

# Moves the class weights to the GPU if available
if torch.cuda.is_available():
    for task_id in task_class_weights:
        task_class_weights[task_id] = task_class_weights[task_id].to("cuda")
    console.print("Class weights moved to GPU", style="green")

In [ ]:
# Custom data collator for MTL that handles multiple label columns
class MTLDataCollatorWithPadding:
    def __init__(self, tokenizer, return_tensors="pt"):
        self.tokenizer = tokenizer
        self.return_tensors = return_tensors
    
    def __call__(self, features):
        # Extracts the standard tokenizer features (input_ids, attention_mask)
        tokenizer_features = []
        label_features = {
            'c2a_labels': [],
            'dbo_labels': [], 
            'vio_labels': []
        }
        
        # Loops through the features and extracts the standard tokenizer features and the MTL label features
        for feature in features:
            # Standard tokenizer inputs
            tokenizer_feature = {
                'input_ids': feature['input_ids'],
                'attention_mask': feature['attention_mask']
            }
            tokenizer_features.append(tokenizer_feature)
            
            # MTL labels for each task
            label_features['c2a_labels'].append(feature['c2a_labels'])
            label_features['dbo_labels'].append(feature['dbo_labels'])
            label_features['vio_labels'].append(feature['vio_labels'])
        
        # Uses the standard data collator for tokenizer inputs
        standard_collator = DataCollatorWithPadding(
            tokenizer=self.tokenizer,
            return_tensors=self.return_tensors
        )
        batch = standard_collator(tokenizer_features)
        
        # Adds the label tensors
        for label_name, label_values in label_features.items():
            batch[label_name] = torch.tensor(label_values, dtype=torch.long)
        
        return batch

In [ ]:
# Creates the data collator for multi-task learning
# (Pads the tokenized inputs to the same length, Stacks the tokenized inputs into a batch, Returns the tokenized inputs as PyTorch tensors)
data_collator = MTLDataCollatorWithPadding(
    tokenizer=processor,
    return_tensors="pt"
)
console.print("MTL data collator configured for PyTorch tensors", style="cyan")

- [Full List of Training Arguments](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments)

In [ ]:
# Builds the run name and directories
dataset_mode = global_config_choices['CHOSEN_DATASET_MODE'] # e.g. 'trial' or 'train'
run_version = global_config_choices['RUN_VERSION_TAG'] # e.g. 'v1'

# Builds the run name e.g. 'train-mtl-moderngbert-mtl-v1'
run_name = f"{cfg['dataset_modes'][dataset_mode]['suffix']}-{cfg['subtask_id']}-{cfg['wandb_run_name_parts']['model_arch']}-{cfg['experiment_type']}"
if run_version:
    run_name += f"-{run_version}"

# Builds the output and logging directory paths
output_dir = project_root_dir / cfg['paths']['output_dir_base'] / cfg['subtask_id'] / run_name
logging_dir = project_root_dir / cfg['paths']['logging_dir_base'] / dataset_mode / run_name

console.print(f"Training run: {run_name}", style="bold yellow")
console.print(f"Early stopping patience: {cfg['callbacks']['early_stopping_patience']}", style="cyan")

# Creates the training arguments (Uses the configs settings)
training_args = TrainingArguments(
    output_dir=str(output_dir),
    logging_dir=str(logging_dir),
    run_name=run_name,
    **cfg['training_arguments']
)

console.print(f"Training arguments: {training_args}", style="cyan")

In [ ]:
# Loads the evaluation metrics from the config
loaded_metrics = {}

for metric in cfg['evaluation']['metrics_to_load']:
    loaded_metrics[metric] = evaluate.load(metric)
for metric in loaded_metrics:
    console.print(f"Loaded metric: {metric}", style="green")

# Replaces the current compute_metrics function with the MTL version
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    # Unpacks the logits for each task
    c2a_logits, dbo_logits, vio_logits = logits
    
    # Extracts the labels for each task
    c2a_labels = labels['c2a_labels']
    dbo_labels = labels['dbo_labels'] 
    vio_labels = labels['vio_labels']
    
    # Converts the logits to predictions
    c2a_preds = np.argmax(c2a_logits, axis=-1)
    dbo_preds = np.argmax(dbo_logits, axis=-1)
    vio_preds = np.argmax(vio_logits, axis=-1)
    
    # Filters out ignored labels (-100) for each task
    def filter_predictions_and_labels(preds, labels):
        """Remove samples where labels == -100 (not relevant for this task)"""
        mask = labels != -100
        return preds[mask], labels[mask]
    
    c2a_preds_filtered, c2a_labels_filtered = filter_predictions_and_labels(c2a_preds, c2a_labels)
    dbo_preds_filtered, dbo_labels_filtered = filter_predictions_and_labels(dbo_preds, dbo_labels)
    vio_preds_filtered, vio_labels_filtered = filter_predictions_and_labels(vio_preds, vio_labels)
    
    # Computes metrics for each task using the HF evaluate library
    metrics = {}
    
    # C2A metrics (binary classification)
    if len(c2a_preds_filtered) > 0:
        metrics.update({
            "c2a_f1-macro": loaded_metrics['f1'].compute(
                predictions=c2a_preds_filtered, 
                references=c2a_labels_filtered, 
                average=cfg['evaluation']['f1_average_type']
            )['f1'],
            "c2a_accuracy": loaded_metrics['accuracy'].compute(
                predictions=c2a_preds_filtered, 
                references=c2a_labels_filtered
            )['accuracy'],
            "c2a_precision-macro": loaded_metrics['precision'].compute(
                predictions=c2a_preds_filtered, 
                references=c2a_labels_filtered, 
                average=cfg['evaluation']['precision_average_type']
            )['precision'],
            "c2a_recall-macro": loaded_metrics['recall'].compute(
                predictions=c2a_preds_filtered, 
                references=c2a_labels_filtered, 
                average=cfg['evaluation']['recall_average_type']
            )['recall']
        })
    
    # DBO metrics (4-class classification)
    if len(dbo_preds_filtered) > 0:
        metrics.update({
            "dbo_f1-macro": loaded_metrics['f1'].compute(
                predictions=dbo_preds_filtered, 
                references=dbo_labels_filtered, 
                average=cfg['evaluation']['f1_average_type']
            )['f1'],
            "dbo_accuracy": loaded_metrics['accuracy'].compute(
                predictions=dbo_preds_filtered, 
                references=dbo_labels_filtered
            )['accuracy'],
            "dbo_precision-macro": loaded_metrics['precision'].compute(
                predictions=dbo_preds_filtered, 
                references=dbo_labels_filtered, 
                average=cfg['evaluation']['precision_average_type']
            )['precision'],
            "dbo_recall-macro": loaded_metrics['recall'].compute(
                predictions=dbo_preds_filtered, 
                references=dbo_labels_filtered, 
                average=cfg['evaluation']['recall_average_type']
            )['recall']
        })
    
    # VIO metrics (binary classification)
    if len(vio_preds_filtered) > 0:
        metrics.update({
            "vio_f1-macro": loaded_metrics['f1'].compute(
                predictions=vio_preds_filtered, 
                references=vio_labels_filtered, 
                average=cfg['evaluation']['f1_average_type']
            )['f1'],
            "vio_accuracy": loaded_metrics['accuracy'].compute(
                predictions=vio_preds_filtered, 
                references=vio_labels_filtered
            )['accuracy'],
            "vio_precision-macro": loaded_metrics['precision'].compute(
                predictions=vio_preds_filtered, 
                references=vio_labels_filtered, 
                average=cfg['evaluation']['precision_average_type']
            )['precision'],
            "vio_recall-macro": loaded_metrics['recall'].compute(
                predictions=vio_preds_filtered, 
                references=vio_labels_filtered, 
                average=cfg['evaluation']['recall_average_type']
            )['recall']
        })
    
    # Computes the overall MTL metrics (average across tasks)
    task_f1_scores = []
    task_accuracies = []
    
    if "c2a_f1-macro" in metrics:
        task_f1_scores.append(metrics['c2a_f1-macro'])
        task_accuracies.append(metrics['c2a_accuracy'])
    if "dbo_f1-macro" in metrics:
        task_f1_scores.append(metrics['dbo_f1-macro'])
        task_accuracies.append(metrics['dbo_accuracy'])
    if "vio_f1-macro" in metrics:
        task_f1_scores.append(metrics['vio_f1-macro'])
        task_accuracies.append(metrics['vio_accuracy'])
    
    # Overall MTL performance (macro average across tasks)
    if task_f1_scores:
        metrics['overall_f1-macro'] = np.mean(task_f1_scores)
        metrics['overall_accuracy'] = np.mean(task_accuracies)
    
    return metrics

In [ ]:
# Clears the GPU cache (Useful when doing multiple subsequent runs on the same machine and with limited computational resources)
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Initializes the MultiTaskFocalLossTrainer that we defined above (Takes the same arguments as the standard Trainer class + task-specific class weights and gamma)
trainer = MultiTaskFocalLossTrainer(
    model=model,
    processing_class=processor,
    data_collator=data_collator,
    args=training_args,
    train_dataset=tokenized_dataset[cfg['dataset_splits']['train']],
    eval_dataset=tokenized_dataset[cfg['dataset_splits']['validation']],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg['callbacks']['early_stopping_patience'])],
    task_class_weights=task_class_weights,
    gamma=2.0
)

console.print(f"Trainer initialized with early stopping patience: {cfg['callbacks']['early_stopping_patience']}", style="green")
console.print(f"Training on {len(tokenized_dataset[cfg['dataset_splits']['train']])} samples", style="cyan")
console.print(f"Validating on {len(tokenized_dataset[cfg['dataset_splits']['validation']])} samples", style="cyan")

In [ ]:
# Starts the Tensorboard Dashboard in the Jupyter Notebook (Requires the Tensorboard IDE extension)
# Otherwise, the Tensorboard Dashboard is also available at http://localhost:6006/
%load_ext tensorboard
%tensorboard --logdir "{str(logging_dir)}"

In [ ]:
# Finishes any existing W&B run
if wandb.run is not None:
    console.print("Finishing previous W&B run...", style="yellow")
    wandb.finish()

# Initializes W&B if we chose to log to W&B in the jupyter widget at the beginning of the notebook
if global_config_choices['LOG_TO_WANDB']:
    console.print("W&B logging enabled. Attempting login...", style="yellow")
    
    # Logs in to W&B and initializes run if successful
    if wandb_utils.login_wandb():
        console.print("W&B login successful.", style="bold green")

        # Initializes the W&B run
        try:
            wandb_run = wandb_utils.init_wandb(
                loaded_config=cfg,
                dataset_mode=global_config_choices['CHOSEN_DATASET_MODE'],
                logging_dir=project_root_dir / cfg['paths']['logging_dir_base'],
                version_tag=global_config_choices['RUN_VERSION_TAG']
            )
            
            # Adds W&B to training args and sets up model watching
            if "wandb" not in training_args.report_to:
                training_args.report_to.append("wandb")
                console.print("W&B added to training args", style="green")
            
            # Sets up model watching
            wandb_run.watch(
                models=model,
                log=cfg['wandb']['watch_model_log'], 
                log_freq=cfg['wandb']['watch_model_log_freq']
            )
            console.print(f"W&B model watching enabled: {cfg['wandb']['watch_model_log']} (freq: {cfg['wandb']['watch_model_log_freq']})", style="cyan")
            console.print(f"W&B run initialized: {wandb_run.name} (ID: {wandb_run.id})", style="bold green")
            
        except Exception as e:
            console.print(f"ERROR: Failed to initialize W&B run: {e}", style="bold red")
            wandb_run = None
    else:
        console.print("W&B login failed. Training will proceed without W&B logging.", style="bold red")
        wandb_run = None
else:
    console.print("W&B logging disabled in the widget.", style="yellow")
    wandb_run = None

In [ ]:
# Creates a jupyter widget button to start the training loop
train_btn = widgets.Button(
    description="Train model",
    icon="play",
    button_style="success",
    tooltip="Start the training loop",
    layout=widgets.Layout(width="160px")
)

# Creates a jupyter widget output to display the training logs
log_out = widgets.Output(
    layout=widgets.Layout(border="1px solid #ccc",
                    max_height="350px",
                    overflow="auto",
                    padding="4px")
)

# Defines the function that is called when the training button is clicked
def _train_model(btn):
    # Disables the training button after starting the training run
    train_btn.disabled = True

    # Clears the log output when starting a new training run
    with log_out:
        clear_output(wait=True)
        console.print("Training started...", style="bold green")

    # Tries to launch the training run
    try:
        training_result = trainer.train()
        with log_out:
            console.print("Training finished successfully!", style="bold green")

            if training_result.metrics:
                console.print("Final metrics:", style="cyan")

                # Loops through the metrics and prints them
                for metric, metric_value in training_result.metrics.items():
                    console.print(f"  {metric}: {metric_value:.4f}", style="white")
    except Exception as e:
        with log_out:
            console.print(f"Training failed with error: {e}", style="bold red")
            raise
    finally:
        # Resets the training button
        train_btn.disabled = False

# Adds the training button to the form
train_btn.on_click(_train_model)

# Displays the training button and log output
display(widgets.VBox([
    widgets.HTML("<h4 style='margin:0 0 8px 0'>Run experiment</h4>"),
    train_btn,
    log_out
]))


In [17]:
# Saves the best model (Do not run this cell before the 'Train model' widget has finished, as it will interrupt the trianing)
trainer.save_model(f"{str(output_dir)}/best_model_{cfg['training_arguments']['metric_for_best_model']}")

In [ ]:
# Saves and uploads a W&B artifact of the finetuned model (Uploads the finetuned model to W&B)
if wandb_run is not None:
    try:
        console.print("Saving model and creating W&B artifact...", style="yellow")
        
        # Builds the artifact name, description, and target path
        artifact_name = cfg['wandb']['artifact']['name_template'].format(
            model_arch=cfg['wandb_run_name_parts']['model_arch'],
            subtask_id=cfg['subtask_id'],
            experiment_type=cfg['experiment_type']
        ) # e.g. "moderngbert-mtl-train"
        
        description = cfg['wandb']['artifact']['description_template'].format(
            model_arch=cfg['wandb_run_name_parts']['model_arch'],
            subtask_name=cfg['subtask_name'],
            experiment_type=cfg['experiment_type'],
            dataset_mode_suffix=cfg['dataset_modes'][global_config_choices['CHOSEN_DATASET_MODE']]['suffix']
        ) # e.g. "Fine-tuned moderngbert for multi-task learning (GermEval 2025)"
        
        # Ensures the target path is in "collection/alias" or "project/collection/alias" format for linking
        target_path = cfg['wandb']['artifact']['portfolio_path_template'].format(
            project=cfg['wandb']['project'],
            model_arch=cfg['wandb_run_name_parts']['model_arch'],
            subtask_id=cfg['subtask_id']
        ) # e.g. "Bachelors-Thesis/moderngbert_mtl_models"
        
        # Prints the artifact name, description, local model path, and target W&B path for linking
        console.print(f"Creating artifact: {artifact_name}", style="cyan")
        console.print(f"Description: {description}", style="cyan")
        console.print(f"Local model path: {output_dir}", style="cyan")
        console.print(f"Target W&B path for linking: {target_path}", style="cyan")
        
        # Saves the model to W&B
        artifact = wandb_utils.save_and_upload_model_to_wandb(
            run=wandb_run,
            name=artifact_name,
            model_type=cfg['wandb']['artifact']['model_type'],
            description=description,
            metadata={
                'model_checkpoint': cfg['model_checkpoint'],
                'subtask_id': cfg['subtask_id'],
                'subtask_name': cfg['subtask_name'],
                'experiment_type': cfg['experiment_type'],
                'dataset_mode': global_config_choices['CHOSEN_DATASET_MODE'],
                'run_name': wandb_run.name,
                'training_args': cfg['training_arguments'],
                'final_metrics': trainer.state.log_history[-1] if trainer.state.log_history else {}
            },
            local_path=str(output_dir), # Path to the local directory containing the model files
            target_path=target_path # Path in W&B to link this artifact version
        )
        
    except Exception as e:
        console.print(f"ERROR: Failed to save model artifact to W&B: {e}", style="bold red")
        console.print("Model was saved locally but W&B artifact creation/linking failed.", style="yellow")
    
    # Finishes the W&B run after the model has been saved
    finally:
        console.print("Finishing W&B run...", style="yellow")
        wandb_run.finish()
        console.print("W&B run finished.", style="green")
else:
    console.print("No W&B run to finish (W&B was disabled or failed to initialize).", style="yellow")

console.print(f"Training completed! Model saved to: {target_path}", style="bold green")